In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

print("Python environment is ready!")

Python environment is ready!


In [2]:
%pip install pandas numpy scipy scikit-learn matplotlib seaborn joblib streamlit

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd

bank_data = pd.read_csv("bank-full.csv", sep=";")

print("Total instances:", bank_data.shape[0])
print("Total input features:", bank_data.shape[1] - 1)
print("Target column:", bank_data.columns[-1])

display(bank_data.head())

Total instances: 45211
Total input features: 16
Target column: y


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


## 4. Dataset Inspection and Data Quality Checks

In [5]:
# Check dataset dimensions
print("Dataset Shape:", bank_data.shape)

# Check column names and data types
print("\nDataset Information:")
bank_data.info()

# Check missing values
print("\nMissing Values:")
print(bank_data.isnull().sum())

# Check duplicate records
print("\nDuplicate Records:", bank_data.duplicated().sum())

# Check target variable distribution
print("\nTarget Distribution:")
print(bank_data["y"].value_counts())

Dataset Shape: (45211, 17)

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  object
 16  y          45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB

Missing Values:
age          0
job          0
mar

## 5. Categorical Feature Analysis

In [7]:
# Identify categorical columns
categorical_cols = bank_data.select_dtypes(include=["object"]).columns

# Display unique values in categorical columns
for col in categorical_cols:
    print(f"\nColumn: {col}")
    print(bank_data[col].value_counts())

# Count 'unknown' values
print("\nUnknown Values:")
for col in categorical_cols:
    count = (bank_data[col] == "unknown").sum()
    print(f"{col}: {count}")


Column: job
job
blue-collar      9732
management       9458
technician       7597
admin.           5171
services         4154
retired          2264
self-employed    1579
entrepreneur     1487
unemployed       1303
housemaid        1240
student           938
unknown           288
Name: count, dtype: int64

Column: marital
marital
married     27214
single      12790
divorced     5207
Name: count, dtype: int64

Column: education
education
secondary    23202
tertiary     13301
primary       6851
unknown       1857
Name: count, dtype: int64

Column: default
default
no     44396
yes      815
Name: count, dtype: int64

Column: housing
housing
yes    25130
no     20081
Name: count, dtype: int64

Column: loan
loan
no     37967
yes     7244
Name: count, dtype: int64

Column: contact
contact
cellular     29285
unknown      13020
telephone     2906
Name: count, dtype: int64

Column: month
month
may    13766
jul     6895
aug     6247
jun     5341
nov     3970
apr     2932
feb     2649
jan     1403

## 6. Feature and Target Separation

In [9]:
# Separate input features and target variable
X = bank_data.drop(columns=["y"])
y = bank_data["y"].map({"no": 0, "yes": 1})

# Verify dimensions
print("Input features shape:", X.shape)
print("Target shape:", y.shape)

# Verify target encoding
print("\nEncoded target distribution:")
print(y.value_counts())

# Check categorical and numerical features
print("\nCategorical features:")
print(X.select_dtypes(include=["object"]).columns.tolist())

print("\nNumerical features:")
print(X.select_dtypes(include=["number"]).columns.tolist())

Input features shape: (45211, 16)
Target shape: (45211,)

Encoded target distribution:
y
0    39922
1     5289
Name: count, dtype: int64

Categorical features:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

Numerical features:
['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']


In [10]:
%pip install --upgrade --force-reinstall "numpy==2.2.6" "scipy==1.15.3" "scikit-learn==1.6.1"

  Using cached numpy-2.2.6-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
  Using cached scipy-1.15.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached scikit_learn-1.6.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (18 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached threadpoolctl-3.8.0-py3-none-any.whl.metadata (29 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
Using cached numpy-2.2.6-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.5 MB)
Using cached scipy-1.15.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (37.3 MB)
Using cached scikit_learn-1.6.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (13.1 MB)
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
Using cached threadpoolctl-3.8.0-py3-none-any.whl (32 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
  Attempting uninstall: thr

In [11]:
import numpy
import scipy
import sklearn

from sklearn.model_selection import train_test_split

print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Import successful")

NumPy: 2.2.6
SciPy: 1.15.3
Scikit-learn: 1.6.1
Import successful


## 7. Train-Test Split

In [13]:
# Import train-test split function
from sklearn.model_selection import train_test_split

# Split dataset into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Display training and testing dimensions
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

# Check class distribution
print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

Training features: (36168, 16)
Testing features: (9043, 16)
Training target: (36168,)
Testing target: (9043,)

Training target distribution:
y
0    0.883018
1    0.116982
Name: proportion, dtype: float64

Testing target distribution:
y
0    0.883003
1    0.116997
Name: proportion, dtype: float64


## 8. Data Preprocessing - Encoding and Scaling

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Identify categorical and numerical columns
categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         categorical_features)
    ]
)

print("Preprocessing pipeline created successfully")

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

Preprocessing pipeline created successfully
Numerical features: 7
Categorical features: 9


## 9. Logistic Regression Model

In [17]:

# Cell 9: Logistic Regression Model

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Create Logistic Regression pipeline
lr_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Train model using training dataset
lr_model.fit(X_train, y_train)

# Predict using testing dataset
lr_predictions = lr_model.predict(X_test)

# Calculate model accuracy
lr_accuracy = accuracy_score(y_test, lr_predictions)

print("Logistic Regression trained successfully")
print("Logistic Regression Accuracy:", round(lr_accuracy, 4))

Logistic Regression trained successfully
Logistic Regression Accuracy: 0.9012


## 10. Decision Tree Classification Model

In [19]:
# Cell 10: Decision Tree Classification Model

from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.base import clone

# Create Decision Tree pipeline
dt_model = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", DecisionTreeClassifier(
        random_state=42
    ))
])

# Train Decision Tree model
dt_model.fit(X_train, y_train)

# Predict using testing dataset
dt_predictions = dt_model.predict(X_test)

# Calculate model accuracy
dt_accuracy = accuracy_score(y_test, dt_predictions)

print("Decision Tree trained successfully")
print("Decision Tree Accuracy:", round(dt_accuracy, 4))


Decision Tree trained successfully
Decision Tree Accuracy: 0.8746


## 11. K-Nearest Neighbors (KNN) Classification

In [21]:
# Cell 11: K-Nearest Neighbors Classification

from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.base import clone

# Create KNN pipeline
knn_model = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", KNeighborsClassifier(
        n_neighbors=5
    ))
])

# Train KNN model
knn_model.fit(X_train, y_train)

# Predict using testing dataset
knn_predictions = knn_model.predict(X_test)

# Calculate model accuracy
knn_accuracy = accuracy_score(y_test, knn_predictions)

print("KNN trained successfully")
print("KNN Accuracy:", round(knn_accuracy, 4))


KNN trained successfully
KNN Accuracy: 0.8962


## 12. Naive Bayes Classification Model

In [23]:
# Cell 12: Naive Bayes Classification Model

# Import required libraries
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score

# Create preprocessing pipeline for Naive Bayes
# GaussianNB requires dense numerical input
nb_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ), categorical_features)
    ],
    sparse_threshold=0
)

# Create Naive Bayes classification pipeline
nb_model = Pipeline(steps=[
    ("preprocessor", nb_preprocessor),
    ("classifier", GaussianNB())
])

# Train model using training dataset
nb_model.fit(X_train, y_train)

# Predict using testing dataset
nb_predictions = nb_model.predict(X_test)

# Calculate model accuracy
nb_accuracy = accuracy_score(y_test, nb_predictions)

# Display model training status and accuracy
print("Naive Bayes trained successfully")
print("Naive Bayes Accuracy:", round(nb_accuracy, 4))


Naive Bayes trained successfully
Naive Bayes Accuracy: 0.8548


## 13. Support Vector Machine (SVM) Classification

In [25]:
# Cell 13: Support Vector Machine Classification

from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.base import clone
from sklearn.metrics import accuracy_score

# Create SVM pipeline
svm_model = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", LinearSVC(
        random_state=42,
        max_iter=5000
    ))
])

# Train SVM model
svm_model.fit(X_train, y_train)

# Predict using testing dataset
svm_predictions = svm_model.predict(X_test)

# Calculate model accuracy
svm_accuracy = accuracy_score(y_test, svm_predictions)

print("SVM trained successfully")
print("SVM Accuracy:", round(svm_accuracy, 4))


SVM trained successfully
SVM Accuracy: 0.9001


## 14. Model Evaluation and Performance Comparison

In [27]:
# Cell 14: Evaluate all five classification models

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

# Collect all trained models
models = {
    "Logistic Regression": lr_model,
    "Decision Tree": dt_model,
    "KNN": knn_model,
    "Naive Bayes": nb_model,
    "SVM": svm_model
}

# Store evaluation results
results = []

for name, model in models.items():

    # Predict class labels
    y_pred = model.predict(X_test)

    # Obtain scores for ROC-AUC
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = model.decision_function(X_test)

    # Calculate six performance metrics
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_score),
        "Precision": precision_score(
            y_test, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_test, y_pred, zero_division=0
        ),
        "F1-score": f1_score(
            y_test, y_pred, zero_division=0
        ),
        "MCC": matthews_corrcoef(y_test, y_pred)
    })

# Create comparison table
comparison_df = pd.DataFrame(results)

# Sort models by F1-score
comparison_df = comparison_df.sort_values(
    by="F1-score",
    ascending=False
).reset_index(drop=True)

print("Performance Comparison of Five ML Models")
display(comparison_df.round(4))

# Save results for README and report
comparison_df.to_csv(
    "model_comparison.csv",
    index=False
)

print("\nComparison saved as model_comparison.csv")


Performance Comparison of Five ML Models


,Model,Accuracy,AUC,Precision,Recall,F1-score,MCC
0,Decision Tree,0.8746,0.7015,0.4649,0.4754,0.4701,0.3990
1,Naive Bayes,0.8548,0.8101,0.4059,0.5198,0.4559,0.3774
2,Logistic Regression,0.9012,0.9056,0.6445,0.3478,0.4518,0.4261
3,KNN,0.8962,0.8277,0.5990,0.3403,0.4340,0.4001
4,SVM,0.9001,0.9072,0.6605,0.3015,0.4140,0.4016



Comparison saved as model_comparison.csv


In [28]:

# Cell 15: Save all five trained ML models

import joblib
import os

# Create folder for trained models
os.makedirs("model", exist_ok=True)

# Save each trained model with preprocessing
joblib.dump(lr_model, "model/logistic_regression.joblib")
joblib.dump(dt_model, "model/decision_tree.joblib")
joblib.dump(knn_model, "model/knn.joblib")
joblib.dump(nb_model, "model/naive_bayes.joblib")
joblib.dump(svm_model, "model/svm.joblib")

print("All five trained models saved successfully!")

print("\nSaved model files:")
print(os.listdir("model"))


All five trained models saved successfully!

Saved model files:
['svm.joblib', 'naive_bayes.joblib', 'logistic_regression.joblib', 'decision_tree.joblib', 'knn.joblib']


In [29]:
# Test loading all five saved models
import joblib

model_files = {
    "Logistic Regression": "model/logistic_regression.joblib",
    "Decision Tree": "model/decision_tree.joblib",
    "KNN": "model/knn.joblib",
    "Naive Bayes": "model/naive_bayes.joblib",
    "SVM": "model/svm.joblib"
}

for name, path in model_files.items():
    try:
        loaded_model = joblib.load(path)
        prediction = loaded_model.predict(X_test.iloc[:5])
        print(name, "- Loaded and predicted successfully:", prediction)
    except Exception as e:
        print(name, "- ERROR:", e)


Logistic Regression - Loaded and predicted successfully: [0 0 0 0 0]
Decision Tree - Loaded and predicted successfully: [0 0 0 0 0]
KNN - Loaded and predicted successfully: [0 0 0 0 0]
Naive Bayes - Loaded and predicted successfully: [0 0 0 0 0]
SVM - Loaded and predicted successfully: [0 0 0 0 0]


In [30]:

# Save testing data for Streamlit application
test_data = X_test.copy()
test_data["y"] = y_test

test_data.to_csv("test_data.csv", index=False)

print("test_data.csv created successfully")
print("Total records:", len(test_data))
print("Total columns:", len(test_data.columns))

test_data.csv created successfully
Total records: 9043
Total columns: 17


In [31]:
app_code = '''
import streamlit as st
import pandas as pd
import joblib
from pathlib import Path
import model_utils  # Required for Naive Bayes model

st.set_page_config(
    page_title="Bank Marketing ML Classifier",
    layout="wide"
)

st.title("BITS Pilani AIMLZG565 - Machine Learning Assignment 2")

st.write("**Name:** Pravin Subhash Marathe")

st.write("**BITS ID:** 2026AK05544")

st.header("Bank Marketing - ML Classification")

BASE = Path(__file__).resolve().parent

models = {
    "Logistic Regression": "logistic_regression.joblib",
    "Decision Tree": "decision_tree.joblib",
    "KNN": "knn.joblib",
    "Naive Bayes": "naive_bayes.joblib",
    "SVM": "svm.joblib"
}

@st.cache_resource
def load_model(filename):
    return joblib.load(BASE / "model" / filename)

model_name = st.selectbox("Select ML Model", list(models))
model = load_model(models[model_name])

st.subheader("Model Performance Comparison")
comparison = pd.read_csv(BASE / "model_comparison.csv")
st.dataframe(comparison, use_container_width=True)

st.subheader("Upload Test Data")
uploaded_file = st.file_uploader("Upload CSV file", type="csv")

if uploaded_file is not None:
    data = pd.read_csv(uploaded_file)
else:
    data = pd.read_csv(BASE / "test_data.csv")

st.write("Preview of Test Data")
st.dataframe(data.head(), use_container_width=True)

X = data.drop(columns=["y"], errors="ignore")

if st.button("Predict"):
    predictions = model.predict(X)
    result = data.copy()
    result["Prediction"] = predictions
    result["Prediction Label"] = result["Prediction"].map(
        {0: "No", 1: "Yes"}
    )

    st.success("Prediction completed successfully!")
    st.dataframe(result.head(100), use_container_width=True)

    st.download_button(
        "Download Predictions",
        result.to_csv(index=False),
        file_name="predictions.csv",
        mime="text/csv"
    )
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("app.py created successfully!")


app.py created successfully!


In [32]:
# Create requirements.txt for Streamlit application

requirements = """streamlit
scikit-learn
numpy
pandas
scipy
joblib
matplotlib
seaborn
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("requirements.txt created successfully!")


requirements.txt created successfully!


In [33]:
import importlib.util

if importlib.util.find_spec("streamlit"):
    print("Streamlit is installed")
else:
    print("Streamlit is NOT installed")

Streamlit is installed


In [34]:
# Launch Streamlit application
import subprocess
import sys

process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "127.0.0.1",
        "--server.headless", "true"
    ],
    stdout=open("streamlit.log", "w"),
    stderr=subprocess.STDOUT
)

print("Streamlit process started:", process.pid)
print("Open http://localhost:8501")


Streamlit process started: 6775
Open http://localhost:8501
